# Xay Dung & Chuan Hoa Bang Ban Ghi Nguon: job_source_records

Notebook nay tap trung xay dung bang quan ly ban ghi thu thap theo tung website nguon **`job_source_records`** trong kien truc Silver Layer:
- Phan biet ro rang giua thuc the tin tuyen dung chuan (`jobs`) va ban ghi vat ly duoc cao tu tung website tuyen dung.
- Quan ly dinh danh ban ghi goc (`source_job_id`), duong dan goc (`source_job_url`), va lich su thoi gian xuat hien (`first_seen_at`, `last_seen_at`).
- Ho tro kien truc Deduplication da san: mot vi tri tuyen dung chuan co the duoc lien ket voi nhieu ban ghi nguon khac nhau (TopCV, TopDev, VietnamWorks).
- Kiem tra nghiem ngat tinh toan ven rang buoc DDL PostgreSQL: khoa chinh `source_record_id`, rang buoc duy nhat `UNIQUE(source_id, source_job_id)`, va cac khoa ngoai tham chieu toi `jobs` va `job_sources`.

In [1]:
import os
import json
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cau hinh hien thi pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Moi truong xu ly bang job_source_records da san sang!')

Moi truong xu ly bang job_source_records da san sang!


## 1. Nap Du Lieu Bronze va Thiet Lap Anh Xa Khoa Ngoai

Dong bo thu tu ban ghi voi bang `jobs` (`02_jobs.ipynb`) va lay ma nguon tu bang `job_sources` (`06_job_sources.ipynb`).

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

df_listings = pd.DataFrame(bronze_listings)
df_details = pd.DataFrame(bronze_details)

# Inner join dong bo dung thu tu nhu 02_jobs.ipynb
df_merged = pd.merge(
    df_listings,
    df_details,
    on=['source', 'source_job_id'],
    how='inner',
    suffixes=('_list', '_detail')
)

# Dong bo job_id (1-indexed)
df_merged['job_id'] = df_merged.index + 1

# Anh xa source_code sang source_id theo bang job_sources
source_map = {'topcv': 1, 'topdev': 2, 'vietnamworks': 3, 'itviec': 4}
df_merged['source_id'] = df_merged['source'].map(source_map)

print(f'Tong so ban ghi dong bo: {len(df_merged):,} ban ghi')
print(f'Toan ven ma nguon source_id: {df_merged["source_id"].notna().sum()} / {len(df_merged)} (100.0% hop le)')

Tong so ban ghi dong bo: 2,327 ban ghi
Toan ven ma nguon source_id: 2327 / 2327 (100.0% hop le)


## 2. Kiem Tra Chat Luong Truong Du Lieu & Rang Buoc Unique

In [3]:
# 1. Kiem tra tinh duy nhat cua cap (source_id, source_job_id)
pairs = list(zip(df_merged['source_id'], df_merged['source_job_id'].astype(str)))
unique_pairs = len(set(pairs))
is_unique_pair = unique_pairs == len(df_merged)

# 2. Kiem tra do dai URL
urls = df_merged['job_url_list'].fillna('').astype(str)
max_url_len = urls.apply(len).max()
avg_url_len = int(urls.apply(len).mean())
empty_urls = (urls == '').sum()

print('--- KIEM TRA CHAT LUONG CAC TRUONG CUA job_source_records ---')
print(f'Tinh duy nhat cua cap (source_id, source_job_id): {is_unique_pair} ({unique_pairs:,} / {len(df_merged):,})')
print(f'So luong URL bi rong: {empty_urls}')
print(f'Do dai URL toi da: {max_url_len} ky tu (tuan thu gioi han VARCHAR(500))')
print(f'Do dai URL trung binh: {avg_url_len} ky tu')

--- KIEM TRA CHAT LUONG CAC TRUONG CUA job_source_records ---
Tinh duy nhat cua cap (source_id, source_job_id): True (2,327 / 2,327)
So luong URL bi rong: 0
Do dai URL toi da: 215 ky tu (tuan thu gioi han VARCHAR(500))
Do dai URL trung binh: 85 ky tu


## 3. Khoi Tao Bang job_source_records Chuan DDL PostgreSQL

Khoi tao bang voi khoa chinh `source_record_id`, cac cot khoa ngoai tham chieu va moc thoi gian `first_seen_at`, `last_seen_at`.

In [4]:
records = []
now_iso = datetime.now().isoformat()

for idx, row in df_merged.iterrows():
    source_record_id = idx + 1
    job_id = row['job_id']
    source_id = row['source_id']
    source_job_id = str(row['source_job_id']).strip()
    source_job_url = str(row['job_url_list'] or '').strip()
    crawled_at = row.get('crawled_at_list') or now_iso
    
    records.append({
        'source_record_id': source_record_id,
        'job_id': job_id,
        'source_id': source_id,
        'source_job_id': source_job_id,
        'source_job_url': source_job_url,
        'first_seen_at': crawled_at,
        'last_seen_at': crawled_at,
        'created_at': now_iso,
        'updated_at': now_iso
    })

df_job_source_records = pd.DataFrame(records)

print('Khoi tao bang job_source_records thanh cong:')
print(f'  - Tong so ban ghi: {len(df_job_source_records):,}')
print(f'  - Tinh duy nhat cua Primary Key source_record_id: {df_job_source_records["source_record_id"].is_unique}')
print(f'  - Toan ven khoa ngoai job_id: {df_job_source_records["job_id"].between(1, len(df_merged)).all()} (100.0% hop le)')
print(f'  - Toan ven khoa ngoai source_id: {df_job_source_records["source_id"].isin([1, 2, 3]).all()} (100.0% hop le)')
print(f'  - Tinh duy nhat cua Unique Constraint (source_id, source_job_id): {len(df_job_source_records) == len(df_job_source_records.drop_duplicates(subset=["source_id", "source_job_id"]))}')

Khoi tao bang job_source_records thanh cong:
  - Tong so ban ghi: 2,327
  - Tinh duy nhat cua Primary Key source_record_id: True
  - Toan ven khoa ngoai job_id: True (100.0% hop le)
  - Toan ven khoa ngoai source_id: True (100.0% hop le)
  - Tinh duy nhat cua Unique Constraint (source_id, source_job_id): True


## 4. Phan Tich Thong Ke & Preview Bang job_source_records

In [5]:
print('--- 1. CAU TRUC BANG job_source_records (INFO) ---')
print(df_job_source_records.info())

print('\n--- 2. PHAN BO BAN GHI THEO NGUON THU THAP (source_id) ---')
id_to_name = {1: 'TopCV', 2: 'TopDev', 3: 'VietnamWorks', 4: 'ITViec'}
df_source_dist = df_job_source_records['source_id'].map(id_to_name).value_counts()
df_source_summary = pd.DataFrame({
    'So ban ghi': df_source_dist,
    'Ty le (%)': (df_source_dist / len(df_job_source_records) * 100).round(2)
})
print(df_source_summary.to_string())

print('\n--- 3. PREVIEW 10 DONG BANG job_source_records (CHUAN POSTGRESQL) ---')
cols_preview = ['source_record_id', 'job_id', 'source_id', 'source_job_id', 'source_job_url', 'first_seen_at']
print(df_job_source_records[cols_preview].head(10).to_string(index=False))

--- 1. CAU TRUC BANG job_source_records (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 2327 entries, 0 to 2326
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   source_record_id  2327 non-null   int64
 1   job_id            2327 non-null   int64
 2   source_id         2327 non-null   int64
 3   source_job_id     2327 non-null   str  
 4   source_job_url    2327 non-null   str  
 5   first_seen_at     2327 non-null   str  
 6   last_seen_at      2327 non-null   str  
 7   created_at        2327 non-null   str  
 8   updated_at        2327 non-null   str  
dtypes: int64(3), str(6)
memory usage: 610.8 KB
None

--- 2. PHAN BO BAN GHI THEO NGUON THU THAP (source_id) ---
              So ban ghi  Ty le (%)
source_id                          
TopCV               1161      49.89
VietnamWorks        1046      44.95
TopDev               120       5.16

--- 3. PREVIEW 10 DONG BANG job_source_records (CHUAN POSTGRESQL)